## BREAST CANCER PREDICTION: COMPREHENSIVE ALGORITHM EXPLORATION

* **BENIGN** - Cells are not cancerous and won't spread.
* **MALIGNANT** - Cells are cancerous and can spread to other tissues and organs (HARMFUL).

**Pipeline Overview:**
1. **Data Ingestion & Preprocessing:** Load CSV data and apply standard scaling.
2. **Algorithm Exploration:** Dynamically test Logistic Regression, KNN, SVC, and Random Forest.
3. **MLflow Tracking:** Log the hyperparameters, cross-validation results, and test metrics for *every* algorithm.
4. **Champion Model Selection:** Automatically identify the algorithm with the highest accuracy.
5. **Model Export:** Serialize the winning model and the data scaler using `joblib` for deployment.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Import all algorithms for exploration
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn
import joblib
import warnings

warnings.filterwarnings('ignore')

### 1. Data Loading and Preprocessing
We load the dataset, map the categorical target variable to binary integers, drop non-predictive columns, and scale the features. Feature scaling is critically important here because distance-based algorithms like KNN and SVC are highly sensitive to unscaled data.

In [3]:
# Load the dataset from CSV
data_path = 'breast_cancer_data.csv' 
df = pd.read_csv(data_path)

In [4]:
df.head()

,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0
1,20.57,17.77,132.90,1326.0,0.08474,0
2,19.69,21.25,130.00,1203.0,0.10960,0
3,11.42,20.38,77.58,386.1,0.14250,0
4,20.29,14.34,135.10,1297.0,0.10030,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   mean_radius      569 non-null    float64
 1   mean_texture     569 non-null    float64
 2   mean_perimeter   569 non-null    float64
 3   mean_area        569 non-null    float64
 4   mean_smoothness  569 non-null    float64
 5   diagnosis        569 non-null    int64  
dtypes: float64(5), int64(1)
memory usage: 26.8 KB


In [ ]:
# Standard preprocessing for typical breast cancer CSVs
if 'id' in df.columns:
    df = df.drop('id', axis=1)
if 'Unnamed: 32' in df.columns: 
    df = df.drop('Unnamed: 32', axis=1)

# Map diagnosis to binary values
if 'diagnosis' in df.columns:
    #df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
    y = df['diagnosis']
    X = df.drop('diagnosis', axis=1)
else:
    y = df['target']
    X = df.drop('target', axis=1)

# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize and fit the Standard Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training data shape: {X_train_scaled.shape}")
print(f"Testing data shape: {X_test_scaled.shape}")

Training data shape: (455, 5)
Testing data shape: (114, 5)


In [11]:
X_train_scaled

array([[-1.07200079, -0.6584246 , -1.0880801 , -0.93927364, -0.13593988],
       [ 1.74874285,  0.06650173,  1.75115682,  1.74555856,  1.27446827],
       [-0.97473376, -0.93112416, -0.99770871, -0.86758911, -0.61351479],
       ...,
       [ 0.39844772,  1.06867262,  0.50751384,  0.24018351,  1.64641135],
       [ 0.85331409, -0.0380331 ,  0.9054796 ,  0.71527488,  1.33397917],
       [-0.91179628, -0.82431683, -0.87666079, -0.84059858,  0.33717171]])

### 2. Multi-Algorithm Exploration & MLflow
We define a dictionary containing our candidate algorithms and their respective hyperparameter grids. We loop through this dictionary, running `GridSearchCV` for each. MLflow records every experiment, allowing us to compare their performance. The script automatically tracks the "Champion" model across all runs.

In [7]:
mlflow.set_experiment("Breast_Cancer_Algorithm_Exploration")

# Define the models and their hyperparameter grids
algorithms = {
    "Logistic_Regression": {
        "model": LogisticRegression(random_state=42, max_iter=10000),
        "params": {'C': [0.01, 0.1, 1, 10, 100]}
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance']}
    },
    "SVC": {
        "model": SVC(probability=True, random_state=42),
        "params": {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']}
    },
    "Random_Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]}
    }
}

# Variables to keep track of the best overall model
champion_name = ""
champion_model = None
champion_accuracy = 0.0

print("Starting Algorithm Exploration...\n" + "-"*40)

for algo_name, config in algorithms.items():
    with mlflow.start_run(run_name=f"Tuning_{algo_name}"):
        print(f"Training and tuning {algo_name}...")
        
        # Initialize GridSearchCV
        grid_search = GridSearchCV(
            estimator=config["model"], 
            param_grid=config["params"], 
            cv=5, 
            n_jobs=-1, 
            scoring='accuracy'
        )
        
        # Fit the model
        grid_search.fit(X_train_scaled, y_train)
        
        # Extract best model and parameters for this specific algorithm
        best_model_for_algo = grid_search.best_estimator_
        best_params_for_algo = grid_search.best_params_
        
        # Evaluate on the test set
        y_pred = best_model_for_algo.predict(X_test_scaled)
        test_accuracy = accuracy_score(y_test, y_pred)
        
        # Log to MLflow
        mlflow.log_params(best_params_for_algo)
        mlflow.log_metric("test_accuracy", test_accuracy)
        mlflow.sklearn.log_model(best_model_for_algo, f"{algo_name}_model")
        
        print(f"[{algo_name}] Best Params: {best_params_for_algo}")
        print(f"[{algo_name}] Test Accuracy: {test_accuracy * 100:.2f}%\n")
        
        # Check if this is the best model overall
        if test_accuracy > champion_accuracy:
            champion_accuracy = test_accuracy
            champion_model = best_model_for_algo
            champion_name = algo_name

print("-" * 40)
print(f"🏆 CHAMPION MODEL: {champion_name} with {champion_accuracy * 100:.2f}% accuracy!")

2026/03/19 14:59:29 INFO mlflow.tracking.fluent: Experiment with name 'Breast_Cancer_Algorithm_Exploration' does not exist. Creating a new experiment.


Starting Algorithm Exploration...
----------------------------------------
Training and tuning Logistic_Regression...


2026/03/19 15:01:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


[Logistic_Regression] Best Params: {'C': 10}
[Logistic_Regression] Test Accuracy: 86.84%

Training and tuning KNN...


2026/03/19 15:02:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


[KNN] Best Params: {'n_neighbors': 7, 'weights': 'distance'}
[KNN] Test Accuracy: 89.47%

Training and tuning SVC...


2026/03/19 15:02:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


[SVC] Best Params: {'C': 1, 'kernel': 'rbf'}
[SVC] Test Accuracy: 87.72%

Training and tuning Random_Forest...


2026/03/19 15:03:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


[Random_Forest] Best Params: {'max_depth': 10, 'n_estimators': 200}
[Random_Forest] Test Accuracy: 92.11%

----------------------------------------
🏆 CHAMPION MODEL: Random_Forest with 92.11% accuracy!


### 3. Model Serialization (Joblib)
Now that we have systematically identified our champion model, we will save it along with the fitted `StandardScaler`. Saving the scaler is mandatory to ensure that any new data fed into the model during production is transformed identically to the training data.

In [8]:
# Define export filenames
model_filename = 'champion_breast_cancer_model.pkl'
scaler_filename = 'champion_standard_scaler.pkl'

# Dump artifacts to disk
joblib.dump(champion_model, model_filename)
joblib.dump(scaler, scaler_filename)

print(f"Success! The champion model ({champion_name}) has been saved to: {model_filename}")
print(f"The required data scaler has been saved to: {scaler_filename}")

# --- Deployment Snippet ---
# When you build your API or consumption layer, load it like this:
# prod_model = joblib.load('champion_breast_cancer_model.pkl')
# prod_scaler = joblib.load('champion_standard_scaler.pkl')
#
# raw_new_data = [[...]] # User input
# scaled_new_data = prod_scaler.transform(raw_new_data)
# prediction = prod_model.predict(scaled_new_data)

Success! The champion model (Random_Forest) has been saved to: champion_breast_cancer_model.pkl
The required data scaler has been saved to: champion_standard_scaler.pkl


In [13]:
import joblib
import numpy as np

def load_and_predict(new_data):
    """
    Loads the saved model and scaler, processes new data, and makes a prediction.
    """
    # 1. Define the paths to your saved files
    model_path = 'champion_breast_cancer_model.pkl'
    scaler_path = 'champion_standard_scaler.pkl'

    try:
        # 2. Load the trained Champion Model and the StandardScaler
        print("Loading model and scaler...")
        model = joblib.load(model_path)
        scaler = joblib.load(scaler_path)
    except FileNotFoundError as e:
        print(f"Error: Could not find the model or scaler file. {e}")
        return

    # 3. Scale the incoming data
    # IMPORTANT: We use .transform() here, NOT .fit_transform()
    # The scaler must apply the exact same mathematical transformation it learned during training.
    print("Scaling new data...")
    scaled_data = scaler.transform(new_data)

    # 4. Make the prediction
    print("Generating prediction...")
    prediction_code = model.predict(scaled_data)[0]
    
    # Optional: Get the confidence score (probability) if the model supports it
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(scaled_data)[0]
        confidence = max(probabilities) * 100
    else:
        confidence = None

    # 5. Interpret the result
    # Assuming 1 = Malignant (M) and 0 = Benign (B) based on our notebook setup
    diagnosis = "Malignant (Harmful)" if prediction_code == 1 else "Benign (Safe)"
    
    print("\n" + "="*40)
    print(f"DIAGNOSIS RESULT: {diagnosis}")
    if confidence:
        print(f"MODEL CONFIDENCE: {confidence:.2f}%")
    print("="*40 + "\n")


if __name__ == "__main__":
    # --- MOCK DATA FOR TESTING ---
    # The Scikit-Learn Breast Cancer dataset requires exactly 30 features.
    # Here is a mock array representing the measurements of a single cell nucleus.
    
    mock_cell_features = np.array([[
        17.99, 10.38, 122.8, 1001.0, 0.1184
    ]])
    
    # Run the function
    load_and_predict(mock_cell_features)

Loading model and scaler...
Scaling new data...
Generating prediction...



DIAGNOSIS RESULT: Benign (Safe)
MODEL CONFIDENCE: 82.50%

